# soplot by example

Every plot below is declared as objects -- a `SoLayer` per mark, an `MDF` per
modifier -- and then built. The declarations are ordinary values: they can be
stored, passed around and reused, which is the point of the package.

Run the cells top to bottom. The data comes from seaborn's sample sets, so this
notebook needs a network connection the first time.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so

from soplot import MDF, SO, Args, KwArgs, SoLayer, add_barlabel

tips = sns.load_dataset('tips')
tips.head()

## A histogram

`SO.histogram` returns a plot with two layers: the bars, and a density curve
over them. Pass `kde_layer=None` for bars alone, or your own `SoLayer` to
draw the density differently.

In [ ]:
SO.histogram(data=tips, feature='total_bill')

## An outlier box

`SO.outlier_box` draws the sample itself and five reference marks: the
interquartile range, the two whisker bounds, the mean and the median. The
whiskers are Tukey's fences clipped to the observed range, so a bound always
sits on a value the data reaches.

In [ ]:
SO.outlier_box(data=tips, feature='total_bill')

## Adding to a plot you already have

`SO.add_layers` and `SO.modify_plot` take a plot and hand back a new one --
`so.Plot` is immutable -- so a declaration can be applied to any plot, not
only to the ones soplot builds.

In [ ]:
scatter = so.Plot(tips, x='total_bill', y='tip')
scatter = SO.add_layers(
    scatter,
    SoLayer(so.Dots(alpha=0.4)),
    SoLayer(so.Line(color='.2'), so.PolyFit(order=1)),
)
SO.modify_plot(
    scatter,
    MDF.Label(title='tip against total bill', x='total bill', y='tip'),
    MDF.Limit(x=(0, 60)),
)

## Labelling the bars

`add_barlabel` works on the drawn figure, not on the declaration: draw with
`.on(figure).plot(pyplot=True)` first, then hand the figure over. The mark has
to be `so.Bar()` -- `so.Bars()` leaves matplotlib no bar containers to read,
so nothing would be labelled.

In [ ]:
figure = plt.figure(figsize=(6, 3), layout='tight')
so.Plot(tips, x='day').add(so.Bar(), so.Hist()).on(figure).plot(pyplot=True)
add_barlabel(figure)

## One variable against several features

`SO.compare_plot` holds one variable on the base axis and puts a feature on the
other, one subfigure per feature. The per-feature arguments are `Args`: one
entry per feature, and the last entry repeats once the entries run out, so a
single entry configures every subplot.

In [ ]:
figure = plt.figure(figsize=(9, 4), layout='tight')
SO.compare_plot(
    data=tips,
    variable='tip',
    features=['total_bill', 'size'],
    sub_figures=figure.subfigures(1, 2),
    base='x',
    layers=Args(Args(SoLayer(so.Dot(), so.Agg('mean')))),
    modifiers=Args(Args(MDF.Facet(col='time'))),
    plot_vars=Args(KwArgs(color='day')),
)
figure

## A box per feature, histogram on top

`SO.multi_outlier_box` pairs each feature's box with a histogram when
`show_hist` says so, and splits the subfigure to hold both.

In [ ]:
figure = plt.figure(figsize=(7, 6), layout='tight')
SO.multi_outlier_box(
    data=tips,
    features=['total_bill', 'tip'],
    sub_figures=figure.subfigures(2, 1),
    show_hist=Args(True),
    box_vars=Args(KwArgs(band_view=False)),
)
figure